# Exercise 4 Digitization and Data Analytics

## Unsupervised Learning with the Iris Data Set

Content

- Fundamental concepts in interaction with Apache Spark
- Working with Jupyter Notebooks
- Usage of basic algorithms (clustering) from the machine learning library spark ML Library

Preparation of Spark
1. Ensure Java 17 is installed and `JAVA_HOME` is set to `/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home`.
2. Restart the notebook kernel before running the next code cell.
3. The following cell clears old Spark environment variables and starts a local SparkSession.


Preparation of your jupyter notebook and the necessary data

1. Prepare a machine learning directory 

2. Download `kmeans_iris_csv_local.ipynb` and `iris.csv` into the `DDA/DDA EXERCISE/Exercise 5` folder.
3. Run the next code cell to clear stale Spark environment variables, set `JAVA_HOME`, and start a local SparkSession.


In [1]:
%matplotlib inline
import os
from pathlib import Path
cwd = Path.cwd()
irisDataPath = cwd / "iris.csv"
if not irisDataPath.exists():
    irisDataPath = cwd / "DDA" / "DDA EXERCISE" / "Exercise 5" / "iris.csv"
print("irisDataPath=", irisDataPath)
if not irisDataPath.exists():
    raise FileNotFoundError(f"iris.csv file not found at {irisDataPath}")
for key in ("SPARK_HOME", "PYSPARK_HOME", "PYSPARK_DRIVER_PYTHON", "PYSPARK_PYTHON"):
    os.environ.pop(key, None)
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
import platform
print("the python version is: " + platform.python_version())
print("SPARK_HOME=", os.environ.get("SPARK_HOME"))
print("JAVA_HOME=", os.environ.get("JAVA_HOME"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("KMeansIris").getOrCreate()


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import DoubleType, StructType, StructField, StringType

In [ ]:
# create explicit spark context 
# spark = SparkSession.builder.appName("PythonKMeansIris").getOrCreate()


1. Read the overview on DataFrames: https://spark.apache.org/docs/latest/sql-programming-guide.html#datasets-and-dataframes 
2. Have a look into the input file first manually (use editor or command on shell). In the following piece of code a DataFrame is generated, and the csv-file is parsed. A VectorAssembler is defined and used to transform the input DataFrame into an dataset. 

In [ ]:
# generate DataFrame, use DataFrameReader class to parse csv input
dataframe = spark.read.csv(irisDataPath, header=True, schema=StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width", DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width", DoubleType(), True),
    StructField("spezies", StringType(), True)
]))
# define a corresponding vector assembler
assembler = VectorAssembler(inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
                             outputCol="features")
dataset = assembler.transform(dataframe)


In [ ]:
#show parameters: 
assembler.params

In [ ]:
assembler.explainParams()

**Exercise:** Examine the data structure of the DataFrame by utilizing the functions 'first', 'head', 'show' and 'printSchema'.

In [ ]:
dataframe.show(10, truncate=False)
dataframe.printSchema()
print("Row count:", dataframe.count())
print("First row:", dataframe.first())


**Exercise:** Prepare the data for plotting using e.g. 'matplotlib' package. Plot a scatterplot of the sepal dimension and a second one for the petal dimension.

**Hint:** Convert Spark DataFrame to Pandas dataframe object for plotting.

In [ ]:
import matplotlib.pyplot as plt
plot_df = dataframe.toPandas()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(plot_df["sepal_length"], plot_df["sepal_width"], c=plot_df["spezies"].astype("category").cat.codes, cmap="viridis", s=35)
axes[0].set_xlabel("sepal_length")
axes[0].set_ylabel("sepal_width")
axes[0].set_title("Sepal dimensions")
axes[1].scatter(plot_df["petal_length"], plot_df["petal_width"], c=plot_df["spezies"].astype("category").cat.codes, cmap="viridis", s=35)
axes[1].set_xlabel("petal_length")
axes[1].set_ylabel("petal_width")
axes[1].set_title("Petal dimensions")
fig.tight_layout()
plt.show()


**Question** How many clusters would you expect in the data set?

Expected number of clusters: 3 (Iris dataset contains three species).


**Exercise:** Define a k-means-Model and fit it on the dataset with your prediction of numbers of clusters using the setK(N) methods, where N sets the number of expected clusters.

In [ ]:
k = 3
kmeans = KMeans(featuresCol="features", predictionCol="prediction", k=k, seed=42)
model = kmeans.fit(dataset)
predictions = model.transform(dataset)
print("Cluster centers:")
for i, center in enumerate(model.clusterCenters()):
    print(f"Center {i}: {center}")
if hasattr(model, "summary") and model.summary is not None:
    print("Within Set Sum of Squared Errors:", model.summary.trainingCost)
else:
    print("Within Set Sum of Squared Errors:", model.computeCost(dataset))
predictions.groupBy("prediction").count().orderBy("prediction").show()


Evaluate the result, e.g. by computing "Within Set Sum of Squared Errors" and have a look to the clusters.

**Exercise:** Compare the result obtained with the k-means algorithm to the ground truth by performing the steps described in the following.

1. Display the cluster centers found with the k-mean algorithm.     
2. Plot your data in the same plot, and visualize the given classes.     
3. Discuss the outcome.

In [ ]:
from pyspark.sql.functions import col, when
predictions = predictions.withColumn(
    "species_id",
    when(col("spezies") == "Iris-setosa", 0)
    .when(col("spezies") == "Iris-versicolor", 1)
    .when(col("spezies") == "Iris-virginica", 2)
    .otherwise(-1)
)
predictions.select("spezies", "species_id", "prediction").show(10, truncate=False)
predictions.groupBy("spezies", "prediction").count().orderBy("spezies", "prediction").show(20, truncate=False)
plot_df = predictions.select("sepal_length", "sepal_width", "spezies", "prediction").toPandas()
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
for species, group in plot_df.groupby("spezies"):
    ax.scatter(group["sepal_length"], group["sepal_width"], label=species, alpha=0.7, s=40)
ax.set_xlabel("sepal_length")
ax.set_ylabel("sepal_width")
ax.set_title("Sepal dimensions by true species")
ax.legend()
plt.show()


**Hints:** Perform a type transformation for the given spezies. Discuss how many colours are appropriate for this analysis.

In [ ]:
spark.stop()